# LactoXAI — explainable ML on human milk single-cell transcriptomes

A lactation clock: predict days postpartum from a milk cell's transcriptome, validated on
held-out mothers, with every explanation tested before it is believed.

**Data:** Single Cell Portal [SCP1671](https://singlecell.broadinstitute.org/single_cell/study/SCP1671)
(Nyquist et al., *PNAS* 2022). Download `expression/`, `metadata/`, `cluster/` and `gene_list/`.

**Runs on:** Kaggle (attach the dataset) or locally (put the download in `./data`).
Outputs go to `/kaggle/working` on Kaggle and `./outputs` locally.

**Install first, in its own cell, then restart the kernel** — never install and import in the same cell:

```
pip install --no-deps scanpy legacy-api-wrap session-info2 natsort pynndescent umap-learn
pip install anndata shap scikit-learn matplotlib pandas scipy h5py
```

The first run streams a ~7 GB text matrix once (15–50 min) and caches an `.h5ad`;
later runs load the cache in seconds.

In [ ]:
# =============================================================================
# LactoXAI — ML + explainability on SCP1671 (human milk scRNA-seq)
# Single-cell version. Run top to bottom.
# =============================================================================
import os, re, json, gzip, glob, warnings, traceback
import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib, matplotlib.pyplot as plt
import scanpy as sc, anndata as ad, shap
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.decomposition import NMF

warnings.filterwarnings("ignore"); sc.settings.verbosity = 0
matplotlib.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170,
    "savefig.bbox": "tight", "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11})

# ------------------------------------------------------------------ config --
ON_KAGGLE = os.path.isdir("/kaggle/input")
OUT = "/kaggle/working" if ON_KAGGLE else os.path.abspath("outputs")
FIGDIR, RESDIR = f"{OUT}/figures", f"{OUT}/results"
CACHE           = f"{RESDIR}/SCP1671_milk.h5ad"
GENE_CHUNK      = 500       # genes per read chunk (~200 MB each)
MIN_CELLS_GENE  = 3
N_HVG           = 1500
SUBSAMPLE_CELLS = 25000      # None = all cells
N_XAI_GENES     = 300        # feature count for the SHAP model
RANDOM_STATE    = 0
os.makedirs(FIGDIR, exist_ok=True); os.makedirs(RESDIR, exist_ok=True)
R = {}

def save(fig, fid):
    fig.savefig(f"{FIGDIR}/{fid}.png"); print("  ->", fid); plt.show()

import time
_T0 = [time.time()]
def step(name):
    now = time.time()
    print(f"\n{'='*70}\n{name}   (+{(now-_T0[0])/60:.1f} min since last step)\n{'='*70}")
    _T0[0] = now

# ----------------------------------------------------------- 1. discovery --

## 1 · FILE DISCOVERY

In [ ]:
step("1 · FILE DISCOVERY")
# Kaggle: attach the dataset. Locally: put the SCP1671 download in ./data or ../data
hits = [h for root in ["/kaggle/input", "data", "../data"]
        for h in glob.glob(f"{root}/**/expression", recursive=True)]
assert hits, "No SCP1671 'expression' folder found under /kaggle/input, ./data or ../data"
ROOT = os.path.dirname(sorted(hits, key=len)[0]); print("ROOT =", ROOT)

def find_file(*pats):
    for pat in pats:
        for m in sorted(glob.glob(f"{ROOT}/**/{pat}", recursive=True)):
            if os.path.isfile(m): return m
            if os.path.isdir(m):
                inner = [f for f in glob.glob(f"{m}/*") if os.path.isfile(f)]
                if inner: return sorted(inner, key=os.path.getsize)[-1]
    return None

P = {"raw":      find_file("MIT_Milk_Study_Raw_counts.txt*"),
     "lognorm":  find_file("MIT_Milk_Study_log_norm_UMI.txt*"),
     "meta":     find_file("MIT_milk_study_metadata.csv*", "*metadata*.csv*"),
     "umap_all": find_file("cluster_umap.txt*"),
     "modules":  find_file("*time_varying_genes_modules.csv*")}
for k, v in P.items():
    print(f"  {k:9s} {os.path.getsize(v)/1e6:8.1f} MB  {v}" if v else f"  {k:9s}        -")

USE_LOGNORM = P["raw"] is None
EXPR = P["lognorm"] if USE_LOGNORM else P["raw"]
assert EXPR, "No expression matrix found."
print(f"\nusing {'LOG-NORMALISED' if USE_LOGNORM else 'RAW COUNTS'}")

# ----------------------------------------------------------- 2. profiling --

## 2 · PROFILING

In [ ]:
step("2 · PROFILING")
opener = lambda p: gzip.open(p, "rt") if p.endswith(".gz") else open(p, "rt")
with opener(EXPR) as f: header = next(f).rstrip("\n")
SEP = "\t" if header.count("\t") > header.count(",") else ","
n_cols = len(header.split(SEP))
# Counting rows reads the whole multi-GB file; skip it when the cache exists
n_rows = (sum(1 for _ in opener(EXPR)) - 1) if not os.path.exists(CACHE) else -1
print(f"  sep={'TAB' if SEP==chr(9) else 'COMMA'}  rows={n_rows}  cols={n_cols}")
print("  orientation is decided at load time by matching names to the metadata")
R["matrix_shape_raw"] = [n_rows, n_cols - 1]

meta_raw = pd.read_csv(P["meta"], sep=None, engine="python", dtype=str)
if str(meta_raw.iloc[0, 0]).strip().upper() == "TYPE":
    meta = meta_raw.iloc[1:].copy(); print("  TYPE row removed")
else:
    meta = meta_raw.copy()
meta = meta.set_index(meta.columns[0]); meta.index.name = "cell"
print(f"  metadata: {meta.shape[0]} cells x {meta.shape[1]} columns")

def pick(cands, cols):
    for pat in cands:
        for c in cols:
            if pat in c.lower(): return c
    return None

COL_DONOR = pick(["donor","subject","participant","biosample","individual"], meta.columns)
COL_TIME  = pick(["day","postpartum","post_partum","time","age_of_infant"], meta.columns)
# Prefer the human-readable ontology label ("macrophage") over the ID ("CL_0000235")
COL_CTYPE = (pick(["ontology_label"], [c for c in meta.columns if "cell_type" in c.lower()])
             or pick(["cell_type","celltype","cluster","annotation"], meta.columns))
COL_SAMP  = pick(["sample","library","biosample"], meta.columns) or COL_DONOR
# --- override here if any of these are wrong -------------------------------
# COL_DONOR, COL_TIME, COL_CTYPE = "donor_id", "time_post_partum_days", "cell_type__ontology_label"
print(f"  donor={COL_DONOR} | time={COL_TIME} | cell_type={COL_CTYPE} | sample={COL_SAMP}")
assert COL_DONOR and COL_CTYPE, "set COL_DONOR / COL_CTYPE by hand"
tnum = pd.to_numeric(meta[COL_TIME], errors="coerce") if COL_TIME else pd.Series(np.nan, index=meta.index)
TIME_NUMERIC = tnum.notna().mean() > 0.5
print(f"  time numeric: {TIME_NUMERIC}" + (f"  range {tnum.min():.0f}-{tnum.max():.0f}" if TIME_NUMERIC else ""))

# ---------------------------------------------------------------- 3. load --

## 3 · LOAD MATRIX

In [ ]:
step("3 · LOAD MATRIX")
if os.path.exists(CACHE):
    adata = ad.read_h5ad(CACHE); print("  loaded cache")
else:
    blocks, major, minor = [], [], None
    # No global dtype here: it would also hit the gene-name column ("A1BG").
    # Each chunk is cast to float32 on the next line instead.
    for i, df in enumerate(pd.read_csv(EXPR, sep=SEP, index_col=0,
                                       chunksize=GENE_CHUNK)):
        if minor is None: minor = list(df.columns)
        major.extend(list(df.index))
        blocks.append(sp.csr_matrix(df.to_numpy(dtype=np.float32)))
        print(f"  chunk {i:4d}  rows {len(major):6d}", end="\r")
    M = sp.vstack(blocks).tocsr(); print(f"\n  stacked {M.shape}")
    # Orientation by name, not by shape: whichever axis matches the metadata
    # barcodes is the cell axis. (Shape fails here: more cells than genes.)
    mi = set(meta.index)
    ov_cols = sum(str(c) in mi for c in minor)
    ov_rows = sum(str(r) in mi for r in major)
    GENES_ARE_ROWS = ov_cols >= ov_rows
    print(f"  barcode overlap: columns {ov_cols}, rows {ov_rows} "
          f"-> genes are {'ROWS' if GENES_ARE_ROWS else 'COLUMNS'}")
    assert max(ov_cols, ov_rows) > 0, "no barcodes match the metadata on either axis"
    X, cells, genes = (M.T.tocsr(), minor, major) if GENES_ARE_ROWS else (M, major, minor)
    X = sp.csr_matrix(X, dtype=np.float32)
    cells = np.array([str(c) for c in cells]); genes = np.array([str(g) for g in genes])

    # Gene filter done directly on the CSR matrix (no scanpy dispatch involved)
    cells_per_gene = np.bincount(X.indices, minlength=X.shape[1])
    keep = cells_per_gene >= MIN_CELLS_GENE
    X, genes = X[:, keep], genes[keep]
    print(f"  genes kept (seen in >= {MIN_CELLS_GENE} cells): {keep.sum()} / {keep.size}")

    obs = meta.reindex(cells)
    print(f"  cells {len(cells)} | with metadata {obs.notna().any(axis=1).sum()}")
    for c in obs.columns:                      # h5ad cannot store mixed str/NaN columns
        if obs[c].dtype == object:
            obs[c] = obs[c].fillna("").astype(str)
    adata = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=pd.Index(genes, name="gene")))
    adata.var_names_make_unique()
    assert adata.n_vars > 1000, f"only {adata.n_vars} genes survived — something is wrong upstream"
    try:
        adata.write_h5ad(CACHE); print("  cached ->", CACHE)
    except Exception as e:
        print("  WARNING: cache not written, continuing without it:", e)

# Always work on a classic scipy csr_matrix; a reloaded h5ad can return other types
adata.X = sp.csr_matrix(adata.X, dtype=np.float32)
print(f"  X: {type(adata.X).__name__} {adata.shape}")
if adata.n_vars < 1000:
    raise SystemExit(f"Cache at {CACHE} is broken ({adata.n_vars} genes). "
                     f"Delete it with os.remove(CACHE) and re-run.")

R["n_cells_loaded"], R["n_genes_loaded"] = int(adata.n_obs), int(adata.n_vars)

adata.obs["donor"]     = adata.obs[COL_DONOR].astype(str)
adata.obs["sample_id"] = adata.obs[COL_SAMP].astype(str)
adata.obs["cell_type"] = adata.obs[COL_CTYPE].astype(str)
if TIME_NUMERIC:
    adata.obs["days"] = pd.to_numeric(adata.obs[COL_TIME], errors="coerce")
else:
    order = sorted(adata.obs[COL_TIME].astype(str).unique())
    adata.obs["days"] = adata.obs[COL_TIME].astype(str).map({v: i for i, v in enumerate(order)}).astype(float)
    print("  stage order:", order)
adata = adata[adata.obs["days"].notna()].copy()
print(f"  cells {adata.n_obs} | donors {adata.obs.donor.nunique()} | types {adata.obs.cell_type.nunique()}")

# ------------------------------------------------------- 4. QC + features --

## 4 · QC AND FEATURES

In [ ]:
step("4 · QC AND FEATURES")
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
Xc = adata.X
adata.obs["n_genes_by_counts"] = np.diff(Xc.indptr)
adata.obs["total_counts"]      = np.asarray(Xc.sum(axis=1)).ravel()
mt_counts = np.asarray(Xc[:, adata.var["mt"].to_numpy()].sum(axis=1)).ravel()
adata.obs["pct_counts_mt"]     = 100 * mt_counts / np.maximum(adata.obs["total_counts"].to_numpy(), 1)
print(f"  mito genes: {int(adata.var['mt'].sum())} | median genes/cell "
      f"{np.median(adata.obs['n_genes_by_counts']):.0f} | median %mt {np.median(adata.obs['pct_counts_mt']):.1f}")
n0 = adata.n_obs
adata = adata[(adata.obs.n_genes_by_counts > 300) & (adata.obs.pct_counts_mt < 25)].copy()
print(f"  QC: {n0} -> {adata.n_obs}")
R["n_cells_after_qc"] = int(adata.n_obs)
if SUBSAMPLE_CELLS and adata.n_obs > SUBSAMPLE_CELLS:
    sc.pp.subsample(adata, n_obs=SUBSAMPLE_CELLS, random_state=RANDOM_STATE)
    print(f"  subsampled to {adata.n_obs}")
if not USE_LOGNORM:
    adata.layers["counts"] = adata.X.copy()
    lib = np.asarray(adata.X.sum(axis=1)).ravel()
    adata.X = sp.csr_matrix(sp.diags(1e4 / np.maximum(lib, 1)) @ adata.X, dtype=np.float32)
    adata.X.data = np.log1p(adata.X.data)
else:
    print("  already log-normalised; skipping normalise")
adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, batch_key="donor")
print("  HVGs:", int(adata.var.highly_variable.sum()))

PROGRAMS = {
    "casein_secretion":    ["CSN1S1","CSN2","CSN3","LALBA","CSN1S2A"],
    "lipid_synthesis":     ["XDH","BTN1A1","PLIN2","FASN","SCD","ADFP"],
    "hmo_glycosylation":   ["B4GALT1","ST3GAL1","FUT2","FUT3","B3GNT2","ST6GAL1"],
    "secretory_machinery": ["SEC61A1","SSR4","XBP1","HSPA5","SEC11C"],
    "stress_response":     ["JUN","FOS","HSPA1A","DDIT3","JUNB"],
    "immune_signalling":   ["CD74","HLA-DRA","B2M","LYZ","IL1B"]}
PRETTY = {k: k.replace("_"," ").capitalize() for k in PROGRAMS}
PRETTY["hmo_glycosylation"] = "HMO glycosylation"

prog_cols = []
for name, gs in PROGRAMS.items():
    present = [g for g in gs if g in adata.var_names]
    if len(present) >= 2:
        sc.tl.score_genes(adata, present, score_name=f"prog_{name}"); prog_cols.append(f"prog_{name}")
    print(f"  {name:22s} {len(present)}/{len(gs)} in dataset")
R["programs_scored"] = prog_cols

# ------------------------------------------------- 5. descriptive figures --

## 5 · DESCRIPTIVE FIGURES

In [ ]:
step("5 · DESCRIPTIVE FIGURES")
piv = (adata.obs.groupby(["donor","days"]).size().rename("n").reset_index()
       .pivot(index="donor", columns="days", values="n").fillna(0))
nd = (piv > 0).sum(0)
fig, ax = plt.subplots(figsize=(min(16, 1.2+0.4*piv.shape[1]), 0.4*piv.shape[0]+2))
im = ax.imshow(np.log1p(piv.values), aspect="auto", cmap="Blues")
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels([f"{c:.0f}" for c in piv.columns], rotation=90, fontsize=7)
ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=7)
ax.set_xlabel("days postpartum"); ax.set_ylabel("donor")
ax.set_title(f"F1 · Each timepoint comes from a median of {nd.median():.0f} donor(s)\n"
             "time and donor are confounded by design", loc="left")
fig.colorbar(im, ax=ax, label="log(1+cells)"); save(fig, "F1_sampling_grid")
R["median_donors_per_timepoint"] = float(nd.median())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col, lab in zip(axes, ["n_genes_by_counts","total_counts","pct_counts_mt"],
                        ["genes per cell","UMIs per cell","% mitochondrial"]):
    ax.boxplot([adata.obs.loc[adata.obs.donor==d, col].values
                for d in sorted(adata.obs.donor.unique())], showfliers=False)
    ax.set_title(lab); ax.set_xlabel("donor"); ax.set_xticklabels([])
    if col != "pct_counts_mt": ax.set_yscale("log")
fig.suptitle("F2 · Quality varies between donors — a route for leakage", y=1.04)
save(fig, "F2_qc_by_donor")

by_t = adata.obs.groupby("days").size()
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.scatter(by_t.index, by_t.values, s=26, color="#2563eb")
ax.set_xlabel("days postpartum"); ax.set_ylabel("cells")
ax.set_title("F3 · Power is not uniform across lactational time", loc="left")
save(fig, "F3_cells_over_time")

comp = adata.obs.groupby(["days","cell_type"]).size().rename("n").reset_index()
comp["frac"] = comp.groupby("days")["n"].transform(lambda s: s/s.sum())
wide = comp.pivot(index="days", columns="cell_type", values="frac").fillna(0).sort_index()
fig, ax = plt.subplots(figsize=(9, 4))
ax.stackplot(wide.index, wide.T.values, labels=wide.columns)
ax.set_xlabel("days postpartum"); ax.set_ylabel("fraction")
ax.legend(fontsize=6, ncol=2, loc="upper right", frameon=False)
ax.set_title("F5 · Cell-type composition shifts across lactation", loc="left")
save(fig, "F5_composition_over_time")

if prog_cols:
    pm = adata.obs.groupby("cell_type")[prog_cols].mean(); v = np.abs(pm.values).max()
    fig, ax = plt.subplots(figsize=(1.8+1.2*len(prog_cols), 0.4*len(pm)+2))
    im = ax.imshow(pm.values, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set_xticks(range(len(prog_cols)))
    ax.set_xticklabels([c.replace("prog_","") for c in prog_cols], rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(pm))); ax.set_yticklabels(pm.index, fontsize=8)
    ax.set_title("F8 · Milk-synthesis programs by cell type", loc="left")
    fig.colorbar(im, ax=ax); save(fig, "F8_program_scores")

## 6 · EMBEDDING

In [ ]:
step("6 · EMBEDDING")
sc.pp.scale(adata, max_value=10, zero_center=False)
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")
try:
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
    sc.tl.umap(adata, random_state=RANDOM_STATE)
    U = adata.obsm["X_umap"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    for ax, key, title in zip(axes, ["cell_type","donor","days"],
                              ["cell type","donor","days postpartum"]):
        if key == "days":
            s_ = ax.scatter(U[:,0], U[:,1], c=adata.obs[key], s=1.6, cmap="viridis")
            fig.colorbar(s_, ax=ax, shrink=0.75)
        else:
            ax.scatter(U[:,0], U[:,1], c=pd.Categorical(adata.obs[key]).codes, s=1.6, cmap="tab20")
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("F4 · If donor structure mirrors cell-type structure, every model below is suspect", y=1.02)
    save(fig, "F4_umap_triptych")
except Exception:
    print("  UMAP skipped:"); traceback.print_exc(limit=1)

# -------------------------------------------------------------- 7. models --

## 7 · MACHINE LEARNING

In [ ]:
step("7 · MACHINE LEARNING")
# Feature set = HVGs plus every curated program gene present in the data, so the
# ablation in section 10 can test each program instead of reporting it n/a.
prog_genes = [g for gs in PROGRAMS.values() for g in gs if g in adata.var_names]
hv  = adata.var_names[adata.var.highly_variable].union(pd.Index(prog_genes))
sub = adata[:, hv]
Xd  = np.asarray(sub.X.todense() if sp.issparse(sub.X) else sub.X, dtype=np.float32)
y   = np.log1p(adata.obs["days"].to_numpy(dtype=float))
groups, gene_names = adata.obs["donor"].to_numpy(), list(hv)
print("  design", Xd.shape, "| donors", len(np.unique(groups)))

# Speed: max_features="sqrt" checks ~40 genes per split instead of all 1,500,
# and max_samples=0.5 trains each tree on half the cells. ~50x faster than the
# defaults, at little cost in accuracy for a model of this size.
rf = lambda n=200: RandomForestRegressor(n_estimators=n, min_samples_leaf=5,
                                         max_features="sqrt", max_samples=0.5,
                                         n_jobs=-1, random_state=RANDOM_STATE)
ns  = min(5, len(np.unique(groups)))
gkf = GroupKFold(n_splits=ns)

yp_d = np.zeros_like(y); fold = []
for k_, (tr_, te_) in enumerate(gkf.split(Xd, y, groups)):
    m_ = rf().fit(Xd[tr_], y[tr_]); yp_d[te_] = m_.predict(Xd[te_])
    fold.append(r2_score(y[te_], yp_d[te_]))
    print(f"  fold {k_+1}/{ns}  held-out donors {len(np.unique(groups[te_]))}  R2 {fold[-1]:+.3f}")
fold = np.array(fold)
mae_d, r2_d = mean_absolute_error(y, yp_d), r2_score(y, yp_d)
print(f"  DONOR-HELD-OUT  MAE {mae_d:.3f}  R2 {r2_d:.3f}")
R["clock_donor_heldout"] = {"mae_log": float(mae_d), "r2": float(r2_d)}
fig, ax = plt.subplots(figsize=(5.6, 5.2))
ax.scatter(np.expm1(y), np.expm1(yp_d), c=pd.Categorical(groups).codes, s=4, cmap="tab20", alpha=.6)
lim = [0, np.expm1(y).max()*1.05]; ax.plot(lim, lim, "--", color="#999", lw=1)
ax.set_xlabel("actual days postpartum"); ax.set_ylabel("predicted")
ax.set_title(f"F9 · Lactation clock, donor-held-out — R² {r2_d:.3f}", loc="left")
save(fig, "F9_clock_predictions")

yp_r = cross_val_predict(rf(), Xd, y, cv=KFold(ns, shuffle=True, random_state=RANDOM_STATE), n_jobs=1)
mae_r, r2_r = mean_absolute_error(y, yp_r), r2_score(y, yp_r); infl = r2_r - r2_d
print(f"  RANDOM SPLIT    MAE {mae_r:.3f}  R2 {r2_r:.3f}   inflation {infl:+.3f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, vals, lab in [(axes[0], [r2_r, r2_d], "R²"), (axes[1], [mae_r, mae_d], "MAE (log days)")]:
    ax.bar(["random\nsplit","donor\nheld-out"], vals, color=["#9ca3af","#2563eb"]); ax.set_ylabel(lab)
    for i, v in enumerate(vals): ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")
fig.suptitle(f"F10 · Donor leakage inflates R² by {infl:.3f}", y=1.03)
save(fig, "F10_leakage_control")
R["clock_random_split"] = {"mae_log": float(mae_r), "r2": float(r2_r)}
R["r2_inflation_from_leakage"] = float(infl)

# ---- F25: does the clock survive inside lactocytes alone? -------------------
# If R2 collapses, the clock was reading cell-type composition; if it holds,
# there is lactocyte-intrinsic temporal signal.
epi_lab = adata.obs["cell_type"].value_counts().index[0]        # largest population
for lab in adata.obs["cell_type"].unique():
    if any(k in lab.lower() for k in ["epithel", "lactocyte", "luminal"]):
        epi_lab = lab; break
em = (adata.obs["cell_type"] == epi_lab).to_numpy()
ge = groups[em]
nse = min(5, len(np.unique(ge)))
yp_e = np.zeros(em.sum()); fold_e = []
for tr_, te_ in GroupKFold(nse).split(Xd[em], y[em], ge):
    m_ = rf().fit(Xd[em][tr_], y[em][tr_]); yp_e[te_] = m_.predict(Xd[em][te_])
    fold_e.append(r2_score(y[em][te_], yp_e[te_]))
r2_e = r2_score(y[em], yp_e)
print(f"  LACTOCYTES ONLY ('{epi_lab}', {em.sum()} cells): R2 {r2_e:.3f}  "
      f"folds {min(fold_e):.2f}–{max(fold_e):.2f}")
fig, ax = plt.subplots(figsize=(6.5, 3.4))
vals = [r2_d, r2_e]; labs = ["all cells", f"{epi_lab} only"]
ax.barh(labs, vals, color=["#9ca3af", "#2563eb"])
for i, v in enumerate(vals): ax.text(v, i, f"  {v:.3f}", va="center")
ax.set_xlabel("R² (donor-held-out)"); ax.set_xlim(0, max(vals) * 1.25)
ax.set_title(f"F25 · The clock {'survives' if r2_e > 0.5 * r2_d else 'collapses'} "
             f"within the '{epi_lab}' population alone", loc="left")
save(fig, "F25_lactocyte_only_clock")
R["clock_lactocyte_only"] = {"label": epi_lab, "n_cells": int(em.sum()),
                             "r2": float(r2_e), "fold_r2": [float(v) for v in fold_e]}

reps = {f"HVG ({len(gene_names)} genes)": Xd, "PCA (40 comps)": adata.obsm["X_pca"][:, :40].astype(np.float32)}
if prog_cols: reps["curated programs"] = adata.obs[prog_cols].to_numpy(dtype=np.float32)
rep = {}
for name, Xr in reps.items():
    s_ = cross_val_score(rf(), Xr, y, cv=gkf, groups=groups, scoring="r2", n_jobs=1)
    rep[name] = (float(np.mean(s_)), float(np.std(s_)))
    print(f"  {name:24s} R2 {np.mean(s_):.3f} ± {np.std(s_):.3f}")
fig, ax = plt.subplots(figsize=(7, 3.4)); ks = list(rep)
ax.barh(ks, [rep[k][0] for k in ks], xerr=[rep[k][1] for k in ks],
        color=["#dc2626" if "curated" in k else "#2563eb" for k in ks])
ax.set_xlabel("R² (donor-held-out)")
ax.set_title("F21 · Curated milk-synthesis programs carry little of the signal", loc="left")
save(fig, "F21_representation_comparison"); R["representation_r2"] = rep

yct = pd.Categorical(adata.obs["cell_type"]).codes
proba = cross_val_predict(RandomForestClassifier(n_estimators=150, min_samples_leaf=3, max_samples=0.5,
    n_jobs=-1, random_state=RANDOM_STATE), Xd, yct, cv=gkf, groups=groups,
    method="predict_proba", n_jobs=1)
conf, acc = proba.max(1), accuracy_score(yct, proba.argmax(1))
fig, ax = plt.subplots(figsize=(6.5, 3.6)); ax.hist(conf, bins=40, color="#2563eb")
ax.set_xlabel("max class probability"); ax.set_ylabel("cells")
ax.set_title(f"F7 · Classifier confidence (accuracy {acc:.3f}) — bimodal means discrete states", loc="left")
save(fig, "F7_classifier_confidence")
R["celltype_accuracy_donor_heldout"], R["median_confidence"] = float(acc), float(np.median(conf))

# --------------------------------------------------------- 8. trust tests --

## 8 · TRUST TESTS

In [ ]:
step("8 · TRUST TESTS")
# Null: permute timepoints across SAMPLES, keeping each sample's cells together.
# A cell-level shuffle destroys all structure and is too easy to beat.
rng = np.random.default_rng(RANDOM_STATE)
samp = adata.obs["sample_id"].to_numpy()
sd = adata.obs.groupby("sample_id")["days"].first()
N_NULL = 20; null = []
for i in range(N_NULL):
    pm = dict(zip(sd.index, rng.permutation(sd.values)))
    yn = np.log1p(np.array([pm[x] for x in samp], dtype=float))
    s_ = cross_val_score(rf(100), Xd, yn, cv=gkf, groups=groups, scoring="r2", n_jobs=1)
    null.append(float(np.mean(s_))); print(f"  permutation {i+1}/{N_NULL}  R2 {np.mean(s_):+.3f}", end="\r")
null = np.array(null)
p_emp = (np.sum(null >= fold.mean()) + 1) / (N_NULL + 1)
print(f"\n  sample-level null {null.mean():+.3f} (max {null.max():+.3f}) | "
      f"observed fold-mean {fold.mean():+.3f} | p = {p_emp:.3f} (smallest possible {1/(N_NULL+1):.3f})")
fig, ax = plt.subplots(figsize=(6.8, 3.6))
ax.hist(null, bins=10, color="#9ca3af", label=f"{N_NULL} sample-level permutations")
ax.axvline(fold.mean(), color="#2563eb", lw=2.5, label=f"observed {fold.mean():.3f}")
ax.set_xlabel("R² (donor-held-out, fold mean)"); ax.legend(frameon=False)
ax.set_title(f"F22 · No permutation of timepoints across samples reaches the observed R² "
             f"(p = {p_emp:.3f})", loc="left")
save(fig, "F22_sample_permutation_null")
R["null_r2_mean"], R["null_r2_max"], R["null_p"] = float(null.mean()), float(null.max()), float(p_emp)

# fold R2 comes from the clock's own CV loop in section 7 — no refit needed
fig, ax = plt.subplots(figsize=(6, 3.4)); ax.bar(range(1, len(fold)+1), fold, color="#2563eb")
ax.axhline(fold.mean(), ls="--", color="#111", lw=1, label=f"mean {fold.mean():.3f}")
ax.set_xlabel("held-out donor fold"); ax.set_ylabel("R²"); ax.legend(frameon=False)
ax.set_title(f"F23 · Fold R² ranges {fold.min():.2f} to {fold.max():.2f}", loc="left")
save(fig, "F23_fold_variance"); R["fold_r2"] = [float(v) for v in fold]

ydon = pd.Categorical(groups).codes
dacc = cross_val_score(RandomForestClassifier(n_estimators=150, max_samples=0.5, n_jobs=-1, random_state=RANDOM_STATE),
    Xd, ydon, cv=KFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="accuracy", n_jobs=1)
chance = pd.Series(ydon).value_counts(normalize=True).max()
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.bar(["donor classifier","chance"], [dacc.mean(), chance], color=["#dc2626","#9ca3af"])
ax.set_ylabel("accuracy")
ax.set_title(f"F24 · Donor identity is {dacc.mean()/chance:.1f}× more predictable than chance", loc="left")
save(fig, "F24_donor_identity")
R["donor_identity_accuracy"], R["donor_identity_chance"] = float(dacc.mean()), float(chance)

# ----------------------------------------------------------------- 9. XAI --

## 9 · EXPLAINABILITY

In [ ]:
step("9 · EXPLAINABILITY")
keep = np.argsort(-np.var(Xd, axis=0))[:N_XAI_GENES]
keep = np.union1d(keep, [gene_names.index(g) for g in prog_genes if g in gene_names])
print(f"  XAI features: {len(keep)} (top-variance genes + all program genes)")
Xx, gx = Xd[:, keep], [gene_names[i] for i in keep]
tr, te = next(gkf.split(Xd, y, groups))
xm = rf(200); xm.fit(Xx[tr], y[tr])
base_r2 = r2_score(y[te], xm.predict(Xx[te]))
print(f"  XAI model ({N_XAI_GENES} genes) R2 held-out: {base_r2:.3f}")
ei = np.random.default_rng(RANDOM_STATE).choice(te, min(800, len(te)), replace=False)
sv = shap.TreeExplainer(xm).shap_values(Xx[ei], check_additivity=False)
ma = np.abs(sv).mean(0); np.save(f"{RESDIR}/shap_values.npy", sv)
json.dump(gx, open(f"{RESDIR}/shap_gene_names.json", "w"))   # column names for shap_values.npy
top = pd.Series(ma, index=gx).sort_values(ascending=False)
R["top_shap_genes"] = list(top.head(30).index)
print("  top genes:", list(top.head(10).index))

fig = plt.figure(figsize=(8, 7))
shap.summary_plot(sv, Xx[ei], feature_names=gx, max_display=30, show=False)
plt.title("F11 · Genes driving the lactation clock", fontsize=11); save(plt.gcf(), "F11_shap_summary")

de = np.expm1(y[ei])
for tag, i in [("earliest", int(np.argmin(de))), ("latest", int(np.argmax(de)))]:
    pd_ = np.expm1(xm.predict(Xx[ei][i:i+1]))[0]
    c = pd.Series(sv[i], index=gx); c = c.reindex(c.abs().sort_values(ascending=False).index).head(14)[::-1]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(c.index, c.values, color=["#dc2626" if v < 0 else "#2563eb" for v in c.values])
    ax.axvline(0, color="#111", lw=.8); ax.set_xlabel("SHAP contribution (log days)")
    ax.set_title(f"F12 · Predicted {pd_:.0f} days, actual {de[i]:.0f} ({tag})", loc="left")
    save(fig, f"F12_shap_waterfall_{tag}")

pi = permutation_importance(xm, Xx[te][:1200], y[te][:1200], n_repeats=5,
                            random_state=RANDOM_STATE, n_jobs=-1)
pis = pd.Series(pi.importances_mean, index=gx).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6.5, 6)); pis.head(25)[::-1].plot.barh(ax=ax, color="#2563eb")
ax.set_xlabel("drop in R² when permuted"); ax.set_title("F14 · Permutation importance", loc="left")
save(fig, "F14_permutation_importance")
ov = len(set(pis.head(25).index) & set(top.head(25).index)); R["shap_perm_overlap_top25"] = int(ov)
print(f"  SHAP/permutation top-25 overlap: {ov}/25")

g2p = {g: p for p, gs in PROGRAMS.items() for g in gs}
pa = {}
for p in PROGRAMS:
    cols = [i for i, g in enumerate(gx) if g2p.get(g) == p]
    if cols: pa[PRETTY[p]] = float(np.abs(sv[:, cols]).sum(1).mean())
pa["(all other genes)"] = float(np.abs(sv[:, [i for i, g in enumerate(gx) if g not in g2p]]).sum(1).mean())
fig, ax = plt.subplots(figsize=(7, 3.8)); s_ = pd.Series(pa).sort_values()
s_.plot.barh(ax=ax, color=["#9ca3af" if "other" in i else "#2563eb" for i in s_.index])
ax.set_xlabel("summed mean |SHAP|")
ax.set_title("F15 · How much of the clock sits in known milk-synthesis programs?", loc="left")
save(fig, "F15_program_attribution"); R["program_attribution"] = pa

# ---- F16: ablation with coverage check and random-gene control band -------

## 10 · ABLATION (F16)

In [ ]:
step("10 · ABLATION (F16)")
cov = {}
for p, gs in PROGRAMS.items():
    inf_ = [g for g in gs if g in gx]; ind = [g for g in gs if g in adata.var_names]
    cov[p] = {"in_features": inf_, "n_in_features": len(inf_),
              "n_in_dataset": len(ind), "n_total": len(gs)}
    print(f"  {p:22s} {len(inf_)}/{len(gs)} features, {len(ind)}/{len(gs)} in data"
          + ("   <-- NOT ABLATABLE" if not inf_ else ""))
    if inf_: print(f"      {inf_}")

tmean = Xx[tr].mean(0)
def ablate(cols):
    Xa = Xx[te].copy(); Xa[:, cols] = tmean[cols]
    return base_r2 - r2_score(y[te], xm.predict(Xa))

abl = {p: (ablate([i for i, g in enumerate(gx) if g in gs]) if any(g in gx for g in gs) else np.nan)
       for p, gs in PROGRAMS.items()}
sizes = [c["n_in_features"] for c in cov.values() if c["n_in_features"]]
k = int(np.median(sizes)) if sizes else 5
rng2 = np.random.default_rng(RANDOM_STATE)
ctrl = np.array([ablate(rng2.choice(len(gx), k, replace=False)) for _ in range(50)])
chi = float(np.percentile(ctrl, 97.5))
print(f"\n  control: 50 random sets of {k} genes, 97.5th pct = {chi:+.5f}")
for p, v in sorted(abl.items(), key=lambda kv: -(kv[1] if kv[1] == kv[1] else -9)):
    tag = "n/a" if v != v else (f"{v:+.5f}  " + ("ABOVE control" if v > chi else "within control"))
    print(f"  {p:22s} {tag}")

plot = {p: v for p, v in abl.items() if v == v}
s_ = pd.Series(plot).sort_values()
names = [f"{PRETTY[p]}  ({cov[p]['n_in_features']}/{cov[p]['n_total']} genes)" for p in s_.index]
win = s_.idxmax()
fig, ax = plt.subplots(figsize=(8.8, 0.52*len(s_)+2.6))
ax.axvspan(min(ctrl.min(), -1e-4), chi, color="#e5e7eb", zorder=0)
bars = ax.barh(names, s_.values, height=.6, zorder=3,
               color=["#2563eb" if p == win else "#9ca3af" for p in s_.index])
ax.axvline(0, color="#111", lw=.9, zorder=4)
span = max(s_.max(), 1e-3)
for b, v in zip(bars, s_.values):
    ax.text(max(v, 0)+span*.015, b.get_y()+b.get_height()/2,
            f"{v:+.4f}" if abs(v) < .01 else f"{v:+.3f}", va="center", fontsize=9,
            color="#111" if v == s_.max() else "#6b7280", zorder=5)
ax.text(chi, len(s_)-.35, "  random-gene control (95%)", fontsize=8.5, color="#6b7280", va="center")
ax.set_xlim(min(s_.min()*1.5, -span*.05), span*1.30)
ax.set_xlabel("R² lost when the program is ablated")
beat = [PRETTY[p] for p in s_.index[::-1] if s_[p] > chi]
inside = [PRETTY[p] for p in s_.index[::-1] if s_[p] <= chi]
ax.set_title(f"F16 · {', '.join(beat) or 'No programme'} beat the random-gene control"
             + (f"; {', '.join(inside)} do not" if inside else ""), fontsize=11, loc="left")
miss = [PRETTY[p] for p, v in abl.items() if v != v]
if miss:
    ax.text(0, -.9, "Not shown (no genes survived feature selection): " + ", ".join(miss),
            fontsize=8, color="#b45309", transform=ax.get_yaxis_transform(), va="top")
save(fig, "F16_program_ablation")
R["program_ablation_r2_loss"] = {kk: (None if v != v else float(v)) for kk, v in abl.items()}
R["ablation_control_p97_5"], R["ablation_coverage"] = chi, cov
R["base_r2_xai_model"] = float(base_r2)

# ---- F18 faithfulness, F19 randomization, F17 counterfactuals -------------

## 11 · XAI SANITY CHECKS

In [ ]:
step("11 · XAI SANITY CHECKS")
os_, or_ = np.argsort(-ma), np.random.default_rng(RANDOM_STATE).permutation(len(gx))
kk_ = [0, 5, 10, 25, 50, 100, 200, len(gx)]
def curve(order):
    out = []
    for n in kk_:
        Xm = Xx[te].copy()
        if n: Xm[:, order[:n]] = tmean[order[:n]]
        out.append(mean_absolute_error(y[te], xm.predict(Xm)))
    return out
cs, cr = curve(os_), curve(or_)
fig, ax = plt.subplots(figsize=(6.8, 4))
ax.plot(kk_, cs, "o-", color="#2563eb", label="SHAP order")
ax.plot(kk_, cr, "o--", color="#9ca3af", label="random order")
ax.set_xscale("symlog"); ax.set_xlabel("genes masked"); ax.set_ylabel("MAE (log days)")
ax.legend(frameon=False)
ax.set_title("F18 · Faithfulness — a steeper blue curve means the attribution is real", loc="left")
save(fig, "F18_faithfulness")
R["faithfulness"] = {"ks": kk_, "shap": [float(v) for v in cs], "random": [float(v) for v in cr]}

rm = rf(150); rm.fit(Xx[tr], np.random.default_rng(1).permutation(y[tr]))
svr = shap.TreeExplainer(rm).shap_values(Xx[ei][:400], check_additivity=False)
mar = np.abs(svr).mean(0); corr = float(np.corrcoef(ma, mar)[0, 1])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(ma, mar, s=7, alpha=.5, color="#2563eb")
axes[0].set_xlabel("mean |SHAP|, trained"); axes[0].set_ylabel("mean |SHAP|, label-shuffled")
axes[0].set_title(f"r = {corr:.3f}")
t1 = pd.Series(ma, index=gx).sort_values(ascending=False).head(15)
t2 = pd.Series(mar, index=gx).sort_values(ascending=False).head(15)
axes[1].barh(range(15), t1.values[::-1], color="#2563eb", label="trained")
axes[1].barh(range(15), -t2.values[::-1], color="#dc2626", label="shuffled")
axes[1].set_yticks(range(15)); axes[1].set_yticklabels(t1.index[::-1], fontsize=7)
axes[1].axvline(0, color="#111", lw=.8); axes[1].legend(frameon=False, fontsize=8)
fig.suptitle(f"F19 · Randomization check — r = {corr:.2f} "
             f"({'attribution may be data-driven' if corr > .5 else 'attribution tracks the model'})", y=1.03)
save(fig, "F19_randomization_check"); R["randomization_corr"] = corr

probe = Xx[ei][:300]; bp = xm.predict(probe); rows = []
for gi in np.argsort(-ma)[:20]:
    Xp = probe.copy(); Xp[:, gi] += np.std(Xx[:, gi])
    rows.append({"gene": gx[gi], "effect_log_days_per_1sd": float(np.mean(xm.predict(Xp) - bp))})
cf = pd.DataFrame(rows).sort_values("effect_log_days_per_1sd")
cf.to_csv(f"{RESDIR}/F17_counterfactuals.csv", index=False)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.barh(cf["gene"], cf["effect_log_days_per_1sd"],
        color=["#dc2626" if v < 0 else "#2563eb" for v in cf["effect_log_days_per_1sd"]])
ax.axvline(0, color="#111", lw=.8); ax.set_xlabel("Δ predicted log-days per +1 SD expression")
ax.set_title("F17 · Counterfactual gene shifts — hypotheses, not causal claims", loc="left")
save(fig, "F17_counterfactuals")

# ---- F20 NMF programs -----------------------------------------------------

## 12 · UNSUPERVISED PROGRAMS (F20)

In [ ]:
step("12 · UNSUPERVISED PROGRAMS (F20)")
try:
    nmf = NMF(n_components=12, init="nndsvda", max_iter=400, random_state=RANDOM_STATE)
    W = nmf.fit_transform(np.clip(Xd, 0, None))
    H = pd.DataFrame(nmf.components_, columns=gene_names)
    tops = {f"NMF{i+1}": list(H.iloc[i].sort_values(ascending=False).head(25).index) for i in range(len(H))}
    done = False
    if P["modules"]:
        mods = pd.read_csv(P["modules"])
        gcol = next((c for c in mods.columns if "gene" in c.lower()), mods.columns[0])
        mcol = next((c for c in mods.columns if any(x in c.lower() for x in ["module","cluster","group"])), None)
        if mcol:
            pub = {str(m): set(g.dropna().astype(str)) for m, g in mods.groupby(mcol)[gcol]}
            J = pd.DataFrame([[len(set(v)&pub[p])/max(1, len(set(v)|pub[p])) for p in pub]
                              for v in tops.values()], index=list(tops), columns=list(pub))
            fig, ax = plt.subplots(figsize=(1.6+.5*J.shape[1], .42*J.shape[0]+2))
            im = ax.imshow(J.values, cmap="Blues", aspect="auto")
            ax.set_xticks(range(J.shape[1])); ax.set_xticklabels(J.columns, rotation=90, fontsize=7)
            ax.set_yticks(range(J.shape[0])); ax.set_yticklabels(J.index, fontsize=8)
            ax.set_title("F20 · NMF programs vs published modules (Jaccard)", loc="left")
            fig.colorbar(im, ax=ax); save(fig, "F20_nmf_vs_published")
            R["max_jaccard_to_published"] = float(J.values.max()); done = True
    if not done:
        cols = H.max(0).sort_values(ascending=False).head(40).index
        fig, ax = plt.subplots(figsize=(9, 5))
        im = ax.imshow(H.loc[:, cols].values, aspect="auto", cmap="Blues")
        ax.set_xticks(range(40)); ax.set_xticklabels(cols, rotation=90, fontsize=6)
        ax.set_ylabel("NMF program"); ax.set_title("F20 · NMF program loadings", loc="left")
        fig.colorbar(im, ax=ax); save(fig, "F20_nmf_loadings")
    pd.Series(tops).to_json(f"{RESDIR}/nmf_programs.json")
except Exception:
    print("  NMF skipped:"); traceback.print_exc(limit=1)

# ------------------------------------------------------------- 13. report --

## 13 · SUMMARY

In [ ]:
step("13 · SUMMARY")
R["figures"] = sorted(os.listdir(FIGDIR))
with open(f"{RESDIR}/results.json", "w") as f:
    json.dump(R, f, indent=2, default=str)

print(f"cells analysed                    {R.get('n_cells_after_qc')}")
print(f"donors                            {adata.obs['donor'].nunique()}")
print(f"median donors per timepoint       {R.get('median_donors_per_timepoint')}")
print()
print(f"clock R2 donor-held-out           {R['clock_donor_heldout']['r2']:+.3f}")
print(f"clock R2 random split             {R['clock_random_split']['r2']:+.3f}")
print(f"  leakage inflation               {R['r2_inflation_from_leakage']:+.3f}")
print(f"clock R2 fold mean (range)        {np.mean(R['fold_r2']):+.3f} ({min(R['fold_r2']):.2f}–{max(R['fold_r2']):.2f})")
print(f"clock R2 lactocytes only          {R['clock_lactocyte_only']['r2']:+.3f}")
print(f"sample-level null                 mean {R['null_r2_mean']:+.3f}, max {R['null_r2_max']:+.3f}, p = {R['null_p']:.3f}")
print(f"donor-identity accuracy           {R['donor_identity_accuracy']:.3f} (chance {R['donor_identity_chance']:.3f})")
print()
print(f"SHAP / permutation top-25 overlap {R['shap_perm_overlap_top25']}/25")
print(f"randomization correlation         {R['randomization_corr']:+.3f}")
print()
print("VERDICTS")
print(f"  clock beats null                {'YES' if R['null_p'] <= 0.05 else 'NO — report as a negative result'}")
print(f"  clock survives in lactocytes    {'YES' if R['clock_lactocyte_only']['r2'] > .5*R['clock_donor_heldout']['r2'] else 'NO — clock reads composition'}")
print(f"  attribution tracks the model    {'YES' if R['randomization_corr'] < .5 else 'NO — XAI layer is suspect'}")
print()
print(f"figures: {len(R['figures'])} in {FIGDIR}")
print(f"numbers: {RESDIR}/results.json")